# mjorbit P4.3: train the bar-pose estimator on a Colab GPU

Before running: **Runtime > Change runtime type > T4 GPU**. Upload `colab_upload/` to Google Drive as `MyDrive/mjorbit_p4/` (folders `data/`, `code/`, `pose_v1/`).

Run the cells in order. Training saves `state.pt` to Drive after every epoch, so if the session disconnects, start a new session and run all cells again: it resumes from the last finished epoch (the upload already holds epoch 1).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; free -g | head -2; df -h /content | tail -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/mjorbit_p4'
!ls -la {D} {D}/data {D}/pose_v1

In [ ]:
# copy the shards to the local disk, check them, unpack (about 10-20 min; skipped if already done in this session)
import os
if not os.path.exists('/content/demos_v1_pack/index.json'):
    !mkdir -p /content/shards /content/demos_v1
    !cp {D}/data/* /content/shards/
    !cd /content/shards && sha256sum -c SHA256SUMS
    !for f in /content/shards/demos_v1_*.tar; do tar -xf $f -C /content/demos_v1 && rm $f; done
    !ls /content/demos_v1 | wc -l

In [ ]:
# unpack the episodes into flat arrays for fast random access (about 23 GB on the local disk)
!mkdir -p /content/code && cp {D}/code/*.py /content/code/
if not os.path.exists('/content/demos_v1_pack/index.json'):
    !cd /content/code && python pack_data.py /content/demos_v1 /content/demos_v1_pack 2
    !rm -rf /content/demos_v1
!ls -la /content/demos_v1_pack

In [ ]:
# train: 20 epochs in total, resuming from Drive. The checkpoint and log go to Drive after every epoch.
!cd /content/code && python train_pose.py --pack /content/demos_v1_pack --out {D}/pose_v1 --epochs 20 --workers 2 2>&1 | tee -a {D}/pose_v1/train_colab.log

In [ ]:
# per-epoch validation error (median position error in mm, overall and while approaching)
import json
for r in json.load(open(f'{D}/pose_v1/log.json')):
    print(r['epoch'], round(r['all']['pos_mm_med'], 1), round(r['approach']['pos_mm_med'], 1), r['min'])